# NLP Project: Decoder-Only Transformer

This notebook builds a decoder-only Transformer for next-token prediction from file inputs.

## 1. Read and Prepare Source A and Source B

- Source A: children book text file
- Source B: news dataset csv file

This section loads both sources, extracts text from the csv, and creates a train/test split for Source B.

In [1]:
%pip install -q torch pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import math
import re
from collections import Counter
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd

# Source A (children book) and Source B (news csv)
children_ds = load_dataset("roneneldan/TinyStories")
children_text_file_path = "roneneldan/TinyStories"
news_csv_path = Path("news_train.csv")

# Read and split Source A (children book)
children_train_raw = " ".join(children_ds["train"]["text"])
children_test_raw = " ".join(children_ds["validation"]["text"])
children_train_raw = re.sub(r"\s+", " ", children_train_raw).strip()
children_test_raw = re.sub(r"\s+", " ", children_test_raw).strip()
children_raw_text = f"{children_train_raw} {children_test_raw}".strip()

news_df = pd.read_csv(news_csv_path)

# Pick a text-like column automatically (first object/string column).
text_columns = news_df['text']
if text_columns.empty:
    raise ValueError("No string/object text column found in news_train.csv")

# Train/test split for Source B.
news_train_df = news_df.sample(frac=0.9, random_state=42)
news_test_df = news_df.drop(news_train_df.index)

news_train_text = " ".join(news_train_df['text'].tolist())
news_test_text = " ".join(news_test_df['text'].tolist())
news_train_text = re.sub(r"\s+", " ", news_train_text).strip()
news_test_text = re.sub(r"\s+", " ", news_test_text).strip()

print(f"Loaded Source A chars: {len(children_raw_text):,} from {children_text_file_path}")
print(f"Source A split -> train chars: {len(children_train_raw):,}, test chars: {len(children_test_raw):,}")
print(f"Loaded Source B rows: {len(news_df):,} from {news_csv_path}")
print(f"Source B text column selected: {text_columns.name}")
print(f"Source B split -> train rows: {len(news_train_df):,}, test rows: {len(news_test_df):,}")

Loaded Source A chars: 20,395,326 from children_book.txt
Source A split -> train chars: 18,355,793, test chars: 2,039,533
Loaded Source B rows: 120,000 from news_train.csv
Source B text column selected: text
Source B split -> train rows: 108,000, test rows: 12,000


## 2. Build Tokenizer and Vocabulary

Tokenizer and vocabulary are built from Source A and Source B training text (not Source B test text).

In [4]:
token_pattern = re.compile(r"\w+|[^\w\s]", re.UNICODE)

# Tokenize train/test splits for Source A
children_train_tokens = token_pattern.findall(children_train_raw.lower())
children_test_tokens = token_pattern.findall(children_test_raw.lower())
news_train_tokens = token_pattern.findall(news_train_text.lower())
news_test_tokens = token_pattern.findall(news_test_text.lower())

special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]

# Build vocabulary only from training sources: A_train + B_train
train_vocab_tokens = children_train_tokens + news_train_tokens
word_counts = Counter(train_vocab_tokens)

max_vocab_size = 20000
most_common_tokens = [token for token, _ in word_counts.most_common(max(0, max_vocab_size - len(special_tokens)))]
vocab = special_tokens + [token for token in most_common_tokens if token not in special_tokens]

stoi = {token: index for index, token in enumerate(vocab)}
itos = {index: token for token, index in stoi.items()}

pad_id = stoi["<pad>"]
bos_id = stoi["<bos>"]
eos_id = stoi["<eos>"]
unk_id = stoi["<unk>"]
vocab_size = len(vocab)


def encode_token_list(token_list):
    return [bos_id] + [stoi.get(token, unk_id) for token in token_list] + [eos_id]

children_train_ids = encode_token_list(children_train_tokens)
children_test_ids = encode_token_list(children_test_tokens)
news_train_ids = encode_token_list(news_train_tokens)
news_test_ids = encode_token_list(news_test_tokens)
mixed_train_ids = children_train_ids + news_train_ids

print(f"Vocabulary size: {vocab_size:,}")
print(f"Source A train tokens: {len(children_train_tokens):,}")
print(f"Source A test tokens: {len(children_test_tokens):,}")
print(f"Source B train tokens: {len(news_train_tokens):,}")
print(f"Source B test tokens: {len(news_test_tokens):,}")

Vocabulary size: 20,000
Source A train tokens: 4,193,696
Source A test tokens: 457,499
Source B train tokens: 5,106,978
Source B test tokens: 566,797


## 3. Create Training and Test Sequences

This section creates loaders for Source A, Source B train/test, and a mixed training stream.

In [5]:
class NextTokenDataset(Dataset):
    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.token_ids) - self.block_size)

    def __getitem__(self, index):
        x = torch.tensor(self.token_ids[index:index + self.block_size], dtype=torch.long)
        y = torch.tensor(self.token_ids[index + 1:index + self.block_size + 1], dtype=torch.long)
        return x, y


def make_loader(token_ids, block_size=128, batch_size=256, shuffle=True):
    dataset = NextTokenDataset(token_ids, block_size)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=True,
                      num_workers=4, pin_memory=True, persistent_workers=True)


block_size = 128
batch_size = 256

loaders = {
    "A_train": make_loader(children_train_ids, block_size=block_size, batch_size=batch_size, shuffle=True),
    "A_test": make_loader(children_test_ids, block_size=block_size, batch_size=batch_size, shuffle=False),
    "B_train": make_loader(news_train_ids, block_size=block_size, batch_size=batch_size, shuffle=True),
    "B_test": make_loader(news_test_ids, block_size=block_size, batch_size=batch_size, shuffle=False),
    "mixed_train": make_loader(mixed_train_ids, block_size=block_size, batch_size=batch_size, shuffle=True),
}

for name, loader in loaders.items():
    print(f"{name}: {len(loader):,} batches")

A_train: 16,381 batches
A_test: 1,786 batches
B_train: 19,948 batches
B_test: 2,213 batches
mixed_train: 36,330 batches


## 4. Define the Decoder-Only Transformer

The model uses causal self-attention, 2 decoder blocks, model dimension 256, and 4 attention heads.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.attn_dropout = dropout

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        context = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_dropout if self.training else 0.0,
            is_causal=True,
        )
        context = context.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(context)


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=256, num_heads=4, num_layers=2, ff_dim=None, dropout=0.1):
        super().__init__()
        if ff_dim is None:
            target_params = 9_000_000
            fixed_params = vocab_size * d_model + block_size * d_model + num_layers * (4 * d_model * d_model)
            ff_dim = max(2048, int((target_params - fixed_params) / (4 * d_model)))

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([
            DecoderBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.dropout = nn.Dropout(dropout)
        self.block_size = block_size
        self.ff_dim = ff_dim

    def forward(self, idx, targets=None):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None:
                top_values, _ = torch.topk(logits, top_k)
                logits[logits < top_values[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 5. Verify Model Size and Parameter Count

This should land near the 9M parameter target for typical text-file vocabularies.

In [ ]:
def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)


## 6. Train in E2 phase

Runs:
- E2: Mixed baseline (A + B together)

The experiment trains a fresh 2-layer decoder-only transformer (M1: ~9M parameters) and reports test losses on A and B_test.

In [ ]:
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

def run_epoch(model, loader, optimizer=None, clip_grad_norm=1.0, max_batches=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_items = 0

    use_cuda_amp = (device.type == "cuda")

    for batch_idx, (inputs, targets) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda_amp):
            _, loss = model(inputs, targets)

        if is_training:
            optimizer.zero_grad(set_to_none=True)
            if use_cuda_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)
                optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        total_items += inputs.size(0)

    return total_loss / max(1, total_items)

def train_for_steps(model, loader, epochs=1, lr=3e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    history = []
    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, loader, optimizer=optimizer)
        history.append(train_loss)
        print(f"  epoch {epoch}: train loss = {train_loss:.4f}")
    return history

def evaluate_model(model, eval_loader, max_batches=500):
    return run_epoch(model, eval_loader, optimizer=None, max_batches=max_batches)

def create_model():
    model_instance = DecoderOnlyTransformer(
        vocab_size=vocab_size,
        block_size=block_size,
        d_model=256,
        num_heads=4,
        num_layers=2,
        ff_dim=None,
        dropout=0.1,
    ).to(device)
    return model_instance

def run_experiment(name, phase_loaders, epochs_per_phase=1, lr=3e-4):
    print(f"\n{name}")
    model_instance = create_model()
    phase_histories = []

    for phase_index, phase_loader in enumerate(phase_loaders, start=1):
        print(f"Phase {phase_index}/{len(phase_loaders)}")
        phase_history = train_for_steps(model_instance, phase_loader, epochs=epochs_per_phase, lr=lr)
        phase_histories.append(phase_history)

    a_test_loss = evaluate_model(model_instance, loaders["A_test"])
    b_test_loss = evaluate_model(model_instance, loaders["B_test"])
    print(f"{name} -> A_test loss: {a_test_loss:.4f}, B_test loss: {b_test_loss:.4f}")

    return {
        "model": model_instance,
        "phase_histories": phase_histories,
        "A_test_loss": a_test_loss,
        "B_test_loss": b_test_loss,
    }

experiment_results = {}

# # E2: Mixed baseline
experiment_results["E2_Mixed"] = run_experiment(
    name="E2: Mixed (Baseline)",
    phase_loaders=[loaders["mixed_train"]],
    epochs_per_phase=2,
    lr=3e-4,
)


In [ ]:
target_params_35m = 35_000_000
d_model = 256
num_layers = 6

fixed_params_35m = vocab_size * d_model + block_size * d_model + num_layers * (4 * d_model * d_model)
ff_dim_35m = max(2048, int((target_params_35m - fixed_params_35m) / (4 * d_model)))

model_35m = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    block_size=block_size,
    d_model=d_model,
    num_heads=8,
    num_layers=num_layers,
    ff_dim=ff_dim_35m,
    dropout=0.1,
).to(device)

param_count_35m = count_parameters(model_35m)
print(f"35M target model -> ff_dim: {ff_dim_35m}, params: {param_count_35m:,} ({param_count_35m / 1_000_000:.2f}M)")

# Memory optimization for 35M run: use smaller micro-batch and shorter sequence length.
train_block_size_35m = 96
train_batch_size_35m = 32
eval_batch_size_35m = 32

if torch.cuda.is_available():
    torch.cuda.empty_cache()

mixed_train_loader_35m = make_loader(
    mixed_train_ids,
    block_size=train_block_size_35m,
    batch_size=train_batch_size_35m,
    shuffle=True,
    )
a_test_loader_35m = make_loader(
    children_test_ids,
    block_size=train_block_size_35m,
    batch_size=eval_batch_size_35m,
    shuffle=False,
    )
b_test_loader_35m = make_loader(
    news_test_ids,
    block_size=train_block_size_35m,
    batch_size=eval_batch_size_35m,
    shuffle=False,
    )

print(
    f"35M loaders -> mixed_train batches: {len(mixed_train_loader_35m):,}, "
    f"A_test batches: {len(a_test_loader_35m):,}, B_test batches: {len(b_test_loader_35m):,}"
)

history_35m = train_for_steps(model_35m, mixed_train_loader_35m, epochs=2, lr=3e-4)
a_test_loss_35m = evaluate_model(model_35m, a_test_loader_35m)
b_test_loss_35m = evaluate_model(model_35m, b_test_loader_35m)
print(f"E2_Mixed_35M -> A_test loss: {a_test_loss_35m:.4f}, B_test loss: {b_test_loss_35m:.4f}")

experiment_results["E2_Mixed_35M"] = {
    "model": model_35m,
    "phase_histories": [history_35m],
    "A_test_loss": a_test_loss_35m,
    "B_test_loss": b_test_loss_35m,
    "target_params": target_params_35m,
    "actual_params": param_count_35m,
    "ff_dim": ff_dim_35m,
    "train_block_size": train_block_size_35m,
    "train_batch_size": train_batch_size_35m,
}

35M target model -> ff_dim: 28635, params: 35,061,686 (35.06M)
35M loaders -> mixed_train batches: 290,643, A_test batches: 14,293, B_test batches: 17,709
  epoch 1: train loss = 4.1364
  epoch 2: train loss = 3.8645
E2_Mixed_35M -> A_test loss: 4.3316, B_test loss: 3.9489


In [ ]:
torch.save(model_35m.state_dict(), "decoder_transformer_mixed_35m.pth")
print("\nModel saved as decoder_transformer_mixed_35m.pth")


Model saved as decoder_transformer_mixed_35m.pth


In [ ]:
target_params_108m = 108_000_000
d_model = 384
num_layers = 12

fixed_params_108m = vocab_size * d_model + block_size * d_model + num_layers * (4 * d_model * d_model)
ff_dim_108m = max(2048, int((target_params_108m - fixed_params_108m) / (4 * d_model)))

model_108m = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    block_size=block_size,
    d_model=d_model,
    num_heads=12,
    num_layers=num_layers,
    ff_dim=ff_dim_108m,
    dropout=0.1,
).to(device)

param_count_108m = count_parameters(model_108m)
print(f"108M target model -> ff_dim: {ff_dim_108m}, params: {param_count_108m:,} ({param_count_108m / 1_000_000:.2f}M)")

# Memory optimization for 108M run: smaller micro-batch and shorter sequence length.
train_block_size_108m = 64
train_batch_size_108m = 8
eval_batch_size_108m = 8

if torch.cuda.is_available():
    torch.cuda.empty_cache()

mixed_train_loader_108m = make_loader(
    mixed_train_ids,
    block_size=train_block_size_108m,
    batch_size=train_batch_size_108m,
    shuffle=True,
    )
a_test_loader_108m = make_loader(
    children_test_ids,
    block_size=train_block_size_108m,
    batch_size=eval_batch_size_108m,
    shuffle=False,
    )
b_test_loader_108m = make_loader(
    news_test_ids,
    block_size=train_block_size_108m,
    batch_size=eval_batch_size_108m,
    shuffle=False,
    )

print(
    f"108M loaders -> mixed_train batches: {len(mixed_train_loader_108m):,}, "
    f"A_test batches: {len(a_test_loader_108m):,}, B_test batches: {len(b_test_loader_108m):,}"
)

history_108m = train_for_steps(model_108m, mixed_train_loader_108m, epochs=2, lr=3e-4)
a_test_loss_108m = evaluate_model(model_108m, a_test_loader_108m, max_batches=200)
b_test_loss_108m = evaluate_model(model_108m, b_test_loader_108m, max_batches=200)
print(f"E2_Mixed_108M -> A_test loss: {a_test_loss_108m:.4f}, B_test loss: {b_test_loss_108m:.4f}")

experiment_results["E2_Mixed_108M"] = {
    "model": model_108m,
    "phase_histories": [history_108m],
    "A_test_loss": a_test_loss_108m,
    "B_test_loss": b_test_loss_108m,
    "target_params": target_params_108m,
    "actual_params": param_count_108m,
    "ff_dim": ff_dim_108m,
    "train_block_size": train_block_size_108m,
    "train_batch_size": train_batch_size_108m,
}

## 6.1 Experiment Results Table (E1, E2, E3)

This section prints a compact table similar to your figure and a detailed table with `A_test` and `B_test` losses.

In [ ]:
# Compact table in the same layout style as the figure.
summary_layout = pd.DataFrame([
    {
        "Model": "M1 (9M params)",
        "E1: A -> B (Simple first)": "Run 1",
        "E2: Mixed (Baseline)": "Run 4",
        "E3: B -> A (Reverse)": "Run 7",
    }
])

# Detailed metrics table using final test losses from each experiment.
summary_metrics = pd.DataFrame([
    {
        "Experiment": "E1_A_to_B",
        "A_test_loss": experiment_results["E1_A_to_B"]["A_test_loss"],
        "B_test_loss": experiment_results["E1_A_to_B"]["B_test_loss"],
    },
    {
        "Experiment": "E2_Mixed",
        "A_test_loss": experiment_results["E2_Mixed"]["A_test_loss"],
        "B_test_loss": experiment_results["E2_Mixed"]["B_test_loss"],
    },
    {
        "Experiment": "E3_B_to_A",
        "A_test_loss": experiment_results["E3_B_to_A"]["A_test_loss"],
        "B_test_loss": experiment_results["E3_B_to_A"]["B_test_loss"],
    },
])

summary_metrics["A_test_loss"] = summary_metrics["A_test_loss"].map(lambda x: round(float(x), 4))
summary_metrics["B_test_loss"] = summary_metrics["B_test_loss"].map(lambda x: round(float(x), 4))

print("Experiment Layout")
display(summary_layout)
print("\nDetailed Final Losses")
display(summary_metrics)

## 7. Generate Text from a Prompt

Generation uses the currently selected experiment model (`active_model`).

In [ ]:
def encode_prompt(prompt):
    prompt_tokens = token_pattern.findall(prompt.lower())
    prompt_ids = [bos_id] + [stoi.get(token, unk_id) for token in prompt_tokens]
    return torch.tensor(prompt_ids, dtype=torch.long).unsqueeze(0)


def decode_ids(token_ids):
    tokens_out = []
    for token_id in token_ids:
        token = itos.get(int(token_id), "<unk>")
        if token in {"<bos>", "<eos>", "<pad>"}:
            continue
        tokens_out.append(token)
    text = " ".join(tokens_out)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    return text


def generate_text(prompt, model_to_use=None, max_new_tokens=50, temperature=1.0, top_k=50):
    selected_model = model_to_use
    selected_model.eval()
    idx = encode_prompt(prompt).to(device)
    generated = selected_model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return decode_ids(generated[0].tolist())

# Example:
# print(generate_text("breaking news today", max_new_tokens=80))
print(generate_text("once upon a time", max_new_tokens=80))

once upon a time there lived a poor man, who had <unk> upon his old hat. you have forgotten that i had left the country one year and four to three, <unk>, and a few hundred thousand pounds in a year! we have seen this since since our father died from the beginning of our death. ’ ‘ ah, how could i then tell him to say to me? ’ said the old man. ‘
